In [ ]:
# ===== 1) SETUP + HELPERS + TRANSFORMS =====
import os, glob, random
import cv2, numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

# macOS OpenMP fix (safe workaround in notebooks)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Constants
IMAGE_SIZE = 1024
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ---------- Adaptive, deterministic color ops (for val/test) ----------
def gray_world_wb(img):
    imgf = img.astype(np.float32)
    mean = imgf.reshape(-1,3).mean(axis=0) + 1e-6
    scale = mean.mean() / mean
    out = np.clip(imgf * scale, 0, 255).astype(np.uint8)
    return out

def adaptive_gamma(img, target_v=0.5, clip=(0.8, 1.3)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2].astype(np.float32)/255.0
    v_mean = float(np.clip(v.mean(), 0.05, 0.95))
    gamma = np.log(max(v_mean,1e-6)) / np.log(max(target_v,1e-6))
    gamma = float(np.clip(gamma, clip[0], clip[1]))
    x = (img.astype(np.float32)/255.0) ** (1.0/gamma)
    return np.clip(x*255.0,0,255).astype(np.uint8)

def adaptive_clahe(img, base_clip=2.0, tile=(8,8)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2]
    v_std = float(v.std())/255.0
    clip_limit = float(np.clip(base_clip + (0.8 - v_std)*1.0, 1.5, 3.0))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    hsv[...,2] = clahe.apply(v)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def adaptive_color_deterministic(img):
    wb  = gray_world_wb(img)
    gam = adaptive_gamma(wb, target_v=0.5, clip=(0.8, 1.3))
    out = adaptive_clahe(gam, base_clip=2.0, tile=(8,8))
    return out

# ---------- Transforms ----------
def get_val_test_transform_adaptive(image_size=IMAGE_SIZE, norm="imagenet"):
    if norm == "imagenet":
        norm_tf = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    elif norm == "minmax":
        norm_tf = A.Normalize(mean=(0,0,0), std=(1,1,1))
    elif norm == "none":
        norm_tf = A.Lambda(image=lambda x, **k: x)
    else:
        raise ValueError("norm must be 'imagenet' | 'minmax' | 'none'")

    return A.Compose([
        A.Resize(image_size, image_size),
        A.Lambda(image=lambda x, **k: adaptive_color_deterministic(x)),  # deterministic per-image color
        norm_tf,
        ToTensorV2()
    ])

def get_train_transform(image_size=IMAGE_SIZE):
    # random color/geometry ONLY for train; no adaptive deterministic color here
    return A.Compose([
        A.Resize(image_size, image_size),
        # color augs (random)
        A.RandomGamma(gamma_limit=(60, 140), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=15, p=0.4),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),
        # geometry augs (random)
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.3),
        A.Transpose(p=0.3),
        # mild structure
        A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.6), p=0.25),
        # normalize for pretrained encoders
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

# ---------- Viz helpers ----------
def denorm_imagenet(t):
    x = t.detach().cpu().permute(1,2,0).numpy()
    x = x * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(x, 0, 1)


In [ ]:
# ===== 2) DATASET + ROBOFLOW DOWNLOAD =====
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Paths
base_dir = "Almons-Trees-8"  # Folder name Roboflow will create
train_img_dir = os.path.join(base_dir, "train")
val_img_dir   = os.path.join(base_dir, "valid")
test_img_dir  = os.path.join(base_dir, "test")

train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_ann_path   = os.path.join(val_img_dir,   "_annotations.coco.json")
test_ann_path  = os.path.join(test_img_dir,  "_annotations.coco.json")

# Download only if missing
if not os.path.exists(base_dir):
    print(f"📥 Dataset not found at '{base_dir}', downloading from Roboflow...")
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    project = rf.workspace("snir5").project("almons-trees")
    version = project.version(8)
    dataset = version.download("coco-segmentation")
    base_dir = dataset.location  # update in case Roboflow renames
else:
    print(f"✅ Dataset already exists at '{base_dir}', skipping download.")

class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform, flip_h=True, flip_v=False):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform
        self.flip_h = flip_h
        self.flip_v = flip_v

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, info["file_name"])

        # load image
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        # build mask
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((info["height"], info["width"]), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        # optional flips
        if self.flip_h:
            image = np.fliplr(image)
            mask  = np.fliplr(mask)
        if self.flip_v:
            image = np.flipud(image)
            mask  = np.flipud(mask)

        # ensure size match
        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        # apply transforms
        augmented = self.transform(image=image, mask=mask)
        image_t = augmented["image"]
        mask_t  = (augmented["mask"] > 0).unsqueeze(0).float()
        return image_t, mask_t


In [ ]:
# ===== 3) LOADERS + PREVIEW =====

# helper: denormalize ImageNet
def denorm_imagenet(tensor_img):
    """Undo ImageNet normalization and convert to numpy [0,1]"""
    mean = np.array(IMAGENET_MEAN).reshape(1, 1, 3)
    std  = np.array(IMAGENET_STD).reshape(1, 1, 3)
    img = tensor_img.permute(1, 2, 0).cpu().numpy()
    img = (img * std) + mean
    return np.clip(img, 0, 1)

# transforms
train_transform = get_train_transform(image_size=IMAGE_SIZE)          
val_transform   = get_val_test_transform_adaptive(image_size=IMAGE_SIZE, norm="imagenet")
test_transform  = get_val_test_transform_adaptive(image_size=IMAGE_SIZE, norm="imagenet")

# datasets
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, train_transform, flip_h=True, flip_v=False)
val_dataset   = COCOSegmentationDataset(val_img_dir,   val_ann_path,   val_transform,   flip_h=True, flip_v=False)
test_dataset  = COCOSegmentationDataset(test_img_dir,  test_ann_path,  test_transform,  flip_h=True, flip_v=False)

# loaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# preview both versions
def show_batch_with_norm(images, masks):
    n = len(images)
    for i in range(n):
        norm_img = denorm_imagenet(images[i])     # undo normalization
        raw_img  = images[i].permute(1,2,0).cpu().numpy()  # still normalized tensor values

        mask = masks[i][0].cpu().numpy()

        plt.figure(figsize=(12, 6))
        
        # Left: denormalized (human-viewable)
        plt.subplot(1, 3, 1)
        plt.imshow(norm_img)
        plt.title("Adaptive Color (denorm)")
        plt.axis("off")

        # Middle: normalized values (model input)
        plt.subplot(1, 3, 2)
        plt.imshow(raw_img)
        plt.title("Normalized Tensor")
        plt.axis("off")

        # Right: mask
        plt.subplot(1, 3, 3)
        plt.imshow(mask, cmap="gray")
        plt.title("Mask")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

# preview a val batch
for images, masks in val_loader:
    show_batch_with_norm(images, masks)
    break


In [ ]:
# === PART 4: MODEL + LOSS + METRICS ===
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# assumes `device` is defined earlier
model_ImNet_Plus = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)  # model has activation=None
        TP = (inputs * targets).sum(dim=(1, 2, 3))
        FP = ((1 - targets) * inputs).sum(dim=(1, 2, 3))
        FN = (targets * (1 - inputs)).sum(dim=(1, 2, 3))
        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma
        return focal_tversky.mean()

loss_fn = FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75)

@torch.no_grad()
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2.0 * inter + eps) / (union + eps)).mean()

@torch.no_grad()
def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean()


In [ ]:
# === PART 5: OPTIMIZER + SCHEDULER + AMP/CLIP + UTILS (CUDA/MPS aware) ===
import torch
import matplotlib.pyplot as plt

# ---- toggles ----
EPOCHS = 20
BASE_LR = 1e-3
SCHEDULER_TYPE = "plateau"   # "plateau" or "cosine"
MAX_NORM = 1.0               # gradient clipping max norm; set None/0 to disable

# ---- backend/AMP detection ----
if device.type == "cuda":
    USE_AMP = True
    backend_name = torch.cuda.get_device_name(0)
elif device.type == "mps":
    USE_AMP = True
    backend_name = "Apple Silicon (MPS)"
else:
    USE_AMP = False
    backend_name = "CPU"

print(f"Backend: {device.type.upper()} — {backend_name} | AMP: {USE_AMP}")

# ---- optimizer ----
optimizer = torch.optim.Adam(model_ImNet_Plus.parameters(), lr=BASE_LR)

# ---- scheduler ----
if SCHEDULER_TYPE == "plateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )
elif SCHEDULER_TYPE == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-5
    )
else:
    raise ValueError("SCHEDULER_TYPE must be 'plateau' or 'cosine'")

# ---- AMP scaler (CUDA or MPS if available) ----
if device.type == "cuda":
    scaler = torch.cuda.amp.GradScaler(enabled=True)
elif device.type == "mps":
    # torch.amp.GradScaler is available on PyTorch 2.x for MPS; fall back if not
    try:
        scaler = torch.amp.GradScaler(enabled=True)
    except Exception:
        scaler = None
        print("Note: GradScaler not available on this PyTorch build for MPS; using unscaled autocast.")
else:
    scaler = None

# ---- history + utils ----
history = {
    'train_loss': [], 'val_loss': [],
    'train_dice': [], 'val_dice': [],
    'train_iou':  [], 'val_iou':  []
}

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss','val_loss','train_dice','val_dice','train_iou','val_iou']
    plt.figure(figsize=(15,10))
    for m in metrics:
        plt.plot(history[m], label=m)
    plt.xlabel("Epoch"); plt.ylabel("Score"); plt.title("Training & Validation Curves")
    plt.legend(); plt.grid(True); plt.show()

def current_lr(optimizer):
    return optimizer.param_groups[0]['lr']


In [ ]:
# === PART 6: TRAIN LOOP (BCE+Dice fp32 loss, robust epoch-level metrics) + CKPT + PLOTS ===
from tqdm import tqdm
import torch.nn as nn
import torch.nn.utils as nn_utils
from contextlib import nullcontext

best_val_dice = 0.0
patience = 5
epochs_no_improve = 0

# ---- BCE + Soft Dice combo (recommended) ----
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1.0, sigmoid=True):
        super().__init__()
        self.smooth = smooth
        self.sigmoid = sigmoid

    def forward(self, logits, targets):
        # logits: (N,1,H,W), targets: (N,1,H,W) in {0,1}
        if self.sigmoid:
            probs = torch.sigmoid(logits)
        else:
            probs = logits
        probs = probs.float()
        targets = targets.float()

        intersection = (probs * targets).sum(dim=(1,2,3))
        p_sum = probs.sum(dim=(1,2,3))
        t_sum = targets.sum(dim=(1,2,3))
        dice = (2.0 * intersection + self.smooth) / (p_sum + t_sum + self.smooth)
        loss = 1.0 - dice
        return loss.mean()

# weights you can tune later (0.5/0.5 is a strong default)
_BCE = nn.BCEWithLogitsLoss()
_DICE = SoftDiceLoss(smooth=1.0, sigmoid=True)
def bce_dice_loss(logits, targets, wbce=0.5, wdice=0.5):
    return wbce * _BCE(logits, targets) + wdice * _DICE(logits, targets)

# ---- AMP setup: CUDA uses autocast+GradScaler, MPS uses autocast only ----
USE_AMP = device.type in ("cuda", "mps")
if device.type == "cuda":
    autocast_ctx = lambda: torch.cuda.amp.autocast()
    scaler = torch.cuda.amp.GradScaler(enabled=True)
elif device.type == "mps":
    autocast_ctx = lambda: torch.amp.autocast(device_type="mps", dtype=torch.float16)
    scaler = None
else:
    autocast_ctx = lambda: nullcontext()
    scaler = None

EPS = 1e-6
THRESH = 0.5

@torch.no_grad()
def epoch_metric_sums(logits, targets, threshold=THRESH):
    """Strict fp32 on CPU for stable Dice/IoU sums."""
    probs = torch.sigmoid(logits.detach().float()).cpu()
    preds = (probs > threshold).float()
    t = targets.detach().float().cpu()
    I = float((preds * t).sum().item())
    P = float(preds.sum().item())
    T = float(t.sum().item())
    return I, P, T

def safe_fp32_loss(logits, targets, wbce=0.5, wdice=0.5):
    """Compute BCE+Dice in fp32 (outside autocast) for stability."""
    with torch.amp.autocast(
        device_type=("cuda" if device.type=="cuda" else ("mps" if device.type=="mps" else "cpu")),
        enabled=False
    ):
        return bce_dice_loss(logits.float(), targets.float(), wbce=wbce, wdice=wdice)

for epoch in range(1, EPOCHS + 1):
    # ---------- TRAIN ----------
    model_ImNet_Plus.train()
    train_loss = 0.0
    tr_I = tr_P = tr_T = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train] LR={current_lr(optimizer):.2e}"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        # forward: AMP for speed, loss in fp32 for stability
        if USE_AMP:
            with autocast_ctx():
                logits = model_ImNet_Plus(images)
            loss = safe_fp32_loss(logits, masks, wbce=0.5, wdice=0.5)
        else:
            logits = model_ImNet_Plus(images)
            loss = bce_dice_loss(logits, masks, wbce=0.5, wdice=0.5)

        if torch.isnan(loss):
            print("⚠️ NaN loss detected in TRAIN — skipping step")
            continue

        # backward
        if scaler is not None:  # CUDA
            scaler.scale(loss).backward()
            if MAX_NORM and MAX_NORM > 0:
                scaler.unscale_(optimizer)
                nn_utils.clip_grad_norm_(model_ImNet_Plus.parameters(), max_norm=MAX_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:  # MPS/CPU
            loss.backward()
            if MAX_NORM and MAX_NORM > 0:
                nn_utils.clip_grad_norm_(model_ImNet_Plus.parameters(), max_norm=MAX_NORM)
            optimizer.step()

        train_loss += float(loss.item())
        I, P, T = epoch_metric_sums(logits, masks)
        tr_I += I; tr_P += P; tr_T += T

    train_dice = (2.0 * tr_I + EPS) / (tr_P + tr_T + EPS) if (tr_P + tr_T) > 0 else 0.0
    train_iou  = (tr_I + EPS) / (tr_P + tr_T - tr_I + EPS) if (tr_P + tr_T - tr_I) > 0 else 0.0

    # ---------- VALIDATE ----------
    model_ImNet_Plus.eval()
    val_loss = 0.0
    va_I = va_P = va_T = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            if USE_AMP:
                with autocast_ctx():
                    logits = model_ImNet_Plus(images)
                loss = safe_fp32_loss(logits, masks, wbce=0.5, wdice=0.5)
            else:
                logits = model_ImNet_Plus(images)
                loss = bce_dice_loss(logits, masks, wbce=0.5, wdice=0.5)

            if torch.isnan(loss):
                print("⚠️ NaN loss detected in VAL — skipping batch")
                continue

            val_loss += float(loss.item())
            I, P, T = epoch_metric_sums(logits, masks)
            va_I += I; va_P += P; va_T += T

    val_dice = (2.0 * va_I + EPS) / (va_P + va_T + EPS) if (va_P + va_T) > 0 else 0.0
    val_iou  = (va_I + EPS) / (va_P + va_T - va_I + EPS) if (va_P + va_T - va_I) > 0 else 0.0

    # ---------- SCHEDULER STEP ----------
    valid_val_batches = max(1, len(val_loader))
    safe_val_loss = val_loss / valid_val_batches if val_loss == val_loss else 1.0
    if SCHEDULER_TYPE == "plateau":
        scheduler.step(safe_val_loss)
    else:
        scheduler.step()

    # ---------- LOG ----------
    avg_train_loss = train_loss / max(1, len(train_loader))
    avg_val_loss   = val_loss   / max(1, len(val_loader))
    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {train_dice:.4f} | Val Dice: {val_dice:.4f} | "
          f"Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f} | "
          f"LR: {current_lr(optimizer):.2e} | "
          f"[Σ train: I={tr_I:.0f},P={tr_P:.0f},T={tr_T:.0f} | Σ val: I={va_I:.0f},P={va_P:.0f},T={va_T:.0f}]")

    # ---------- CHECKPOINT ----------
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_ImNet_Plus.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': val_dice,
            'val_loss': avg_val_loss
        }, "best_model.pth")
        print("✅ Saved new best model")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve} epoch(s)")

    # ---------- EARLY STOP ----------
    if epochs_no_improve >= patience:
        print("⛔ Early stopping triggered")
        break

    # ---------- HISTORY ----------
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(float(train_dice))
    history['val_dice'].append(float(val_dice))
    history['train_iou'].append(float(train_iou))
    history['val_iou'].append(float(val_iou))

# ---------- PLOTS ----------
plot_learning_curves(history)
2

In [ ]:
# === PART 7: INFERENCE & VISUALIZATION (TEST SET) ===
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from contextlib import nullcontext

# --- config ---
TEST_IMAGE_SIZE = IMAGE_SIZE  # keep same size as training/val
THRESH = 0.5                  # you can sweep later
CKPT_PATH = "best_model.pth"  # path to your saved checkpoint

# --- transform for test (deterministic, per-image adaptive color + chosen norm) ---
test_transform = get_val_test_transform_adaptive(image_size=TEST_IMAGE_SIZE, norm="imagenet")

# --- test dataset/loader (set flips to match what you used for val) ---
test_img_dir = os.path.join(base_dir, "test") if "base_dir" in globals() else "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test"
test_ann_path = os.path.join(test_img_dir, "_annotations.coco.json")
test_dataset = COCOSegmentationDataset(
    img_dir=test_img_dir,
    ann_path=test_ann_path,
    transform=test_transform,
    flip_h=True,   # << set these to SAME values you used in val_dataset
    flip_v=False
)

# --- model (same arch as training) ---
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# --- load checkpoint ---
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# --- denorm helper consistent with ImageNet normalization ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
def denormalize_imagenet(t):
    if isinstance(t, torch.Tensor):
        x = t.detach().cpu().permute(1,2,0).numpy()
    else:
        x = t
    x = x * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    x = np.clip(x, 0, 1)
    return (x * 255).astype(np.uint8)

# --- autocast context for eval (CUDA/MPS safe) ---
if device.type == "cuda":
    eval_autocast = lambda: torch.cuda.amp.autocast()
elif device.type == "mps":
    eval_autocast = lambda: torch.amp.autocast(device_type="mps", dtype=torch.float16)
else:
    eval_autocast = lambda: nullcontext()

# --- visualization ---
def visualize_predictions(model, dataset, device, max_samples=10, thresh=THRESH):
    n = min(len(dataset), max_samples)
    for i in tqdm(range(n), desc="Predicting"):
        image_t, mask_t = dataset[i]            # image_t: (3,H,W) normalized tensor; mask_t: (1,H,W)
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            with eval_autocast():
                logits = model(image_b)
        prob = torch.sigmoid(logits)[0,0].cpu().numpy()
        pred_mask = (prob > thresh).astype(np.uint8)

        img_vis = denormalize_imagenet(image_t)
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        # plots: Original / GT / Pred / Overlay
        fig, axs = plt.subplots(1, 4, figsize=(16, 4))
        axs[0].imshow(img_vis);      axs[0].set_title("Image"); axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray");  axs[1].set_title("Ground Truth"); axs[1].axis("off")
        axs[2].imshow(pred_mask, cmap="gray");axs[2].set_title("Predicted"); axs[2].axis("off")
        overlay = img_vis.copy()
        overlay[ pred_mask.astype(bool) ] = (overlay[ pred_mask.astype(bool) ] * 0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
        axs[3].imshow(overlay);      axs[3].set_title("Overlay"); axs[3].axis("off")
        plt.tight_layout(); plt.show()

# --- run ---
visualize_predictions(model, test_dataset, device, max_samples=10, thresh=THRESH)
